In [ ]:
import json
import glob
import os
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def get_metric_data(file, metric_name, prefix):
    with open(file) as f:
        data = json.load(f)
    
    metric_dict = data.get(metric_name, {})
    res = {}
    
    for k, v in metric_dict.items():
        parts = k.split('/')
        if len(parts) == 3 and parts[0] == prefix and parts[1] == 'Eval' and parts[2].startswith('Exp'):
            try:
                exp_idx = int(parts[2].replace('Exp', '')) - 1
                res[exp_idx] = v
            except:
                continue
    
    sorted_keys = sorted(res.keys())
    return [res[k] for k in sorted_keys]

In [ ]:
def plot_dataset_results(dataset_name, json_files, axes_row):
    metrics_to_plot = [
        {'name': 'Acc', 'prefix': 'Acc', 'label': 'Accuracy (%)', 'scale': 100},
        {'name': 'Loss', 'prefix': 'Loss', 'label': 'Loss', 'scale': 1},
        {'name': 'Forgetting', 'prefix': 'Forgetting', 'label': 'Forgetting', 'scale': 1}
    ]
    
    for i, m in enumerate(metrics_to_plot):
        ax = axes_row[i]
        all_runs_data = []
        
        for f in json_files:
            run_data = get_metric_data(f, m['name'], m['prefix'])
            if not run_data: continue
            
            run_data = np.array(run_data) * m['scale']
            all_runs_data.append(run_data)
            
            run_id = f.split('_run')[-1].split('.')[0]
            ax.plot(run_data, linestyle='--', alpha=0.3)
        
        if all_runs_data:
            avg_data = np.mean(all_runs_data, axis=0)
            ax.plot(avg_data, marker='o', linewidth=2, color='black', label='Average')
            
        ax.set_title(f"{dataset_name.upper()} - {m['name']}")
        ax.set_xlabel("Session")
        ax.set_ylabel(m['label'])
        ax.grid(True, alpha=0.3)
        if i == 0: ax.legend()

In [ ]:
datasets = ['cifar100', 'cub200']
fig, axes = plt.subplots(len(datasets), 3, figsize=(20, 6 * len(datasets)))

for idx, ds in enumerate(datasets):
    ds_files = sorted(glob.glob(f'temp_model_{ds}_*.json'))
    if not ds_files:
        print(f"未找到 {ds} 的结果文件")
        continue
    print(f"加载 {ds}: {len(ds_files)} 个文件")
    plot_dataset_results(ds, ds_files, axes[idx])

plt.tight_layout()
plt.show()